In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_ecobici")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_ecobici = spark.read.table(f"{catalogo}.{esquema_source}.ecobici")

In [0]:
df_ecobici_silver = df_ecobici \
    .withColumn(
        "genero_norm",
        when(lower(col("genero")) == "hombre", "H")
        .when(lower(col("genero")) == "mujer", "F")
        .otherwise("O")
    ) \
    .groupBy("fecha") \
    .agg(
        sum(when(col("genero_norm") == "H", col("viajes")).otherwise(0)).alias("viajes_hombres"),
        sum(when(col("genero_norm") == "F", col("viajes")).otherwise(0)).alias("viajes_mujeres"),
        sum(when(col("genero_norm") == "O", col("viajes")).otherwise(0)).alias("viajes_otros"),
        sum("viajes").alias("total_viajes")
    ) \
    .withColumnRenamed("fecha", "date") \
    .withColumn("ingestion_date", current_timestamp())

In [0]:
df_ecobici_silver.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalogo}.{esquema_sink}.ecobici")

In [0]:
df_weather = spark.read.table(f"{catalogo}.{esquema_source}.weather")

In [0]:
df_weather_silver = df_weather \
    .withColumnRenamed("temperature_max", "temp_max") \
    .withColumnRenamed("temperature_avg", "temp_avg") \
    .withColumnRenamed("precipitation", "lluvia") \
    .withColumn("ingestion_date", current_timestamp())

In [0]:
df_weather_silver.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalogo}.{esquema_sink}.weather")

In [0]:
%sql
SELECT COUNT(*) AS total_ecobici
FROM catalog_ecobici.silver.ecobici;

In [0]:
%sql
SELECT COUNT(*) AS total_weather
FROM catalog_ecobici.silver.weather;

In [0]:
%sql
SELECT date, COUNT(*) AS duplicados
FROM catalog_ecobici.silver.ecobici
GROUP BY date
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT *
FROM catalog_ecobici.silver.ecobici
WHERE total_viajes != (viajes_hombres + viajes_mujeres + viajes_otros);

In [0]:
%sql
SELECT *
FROM catalog_ecobici.silver.weather
WHERE temp_avg IS NULL
   OR temp_max IS NULL
   OR lluvia IS NULL;

In [0]:
%sql
SELECT 
    e_min, e_max, w_min, w_max
FROM (
    SELECT 
        MIN(date) e_min, MAX(date) e_max
    FROM catalog_ecobici.silver.ecobici
) e
CROSS JOIN (
    SELECT 
        MIN(date) w_min, MAX(date) w_max
    FROM catalog_ecobici.silver.weather
) w;